# Model 3: Supplier performance scoring

Synthetic-data development experiment. Read the system guide before interpreting results. These notebooks do not verify or replace your existing Models 1 and 2. CPU runtime is sufficient.

Run cells in order. Outputs are saved to a new timestamped Drive folder.


## 1. Dependencies
Restart the runtime if requested after installation. The final cell saves the exact installed versions for reproducibility.


In [ ]:
%pip install -q "pandas>=2,<3" "numpy>=1.26,<3" "scikit-learn>=1.4,<2" joblib matplotlib


## 2. Mount Drive and locate the original dataset
Upload the original CSV to `MyDrive/PharmacyDSS/data/raw/`. Edit ROOT if your folder differs. No input files are modified.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
from datetime import datetime, timezone
import subprocess, sys, json
import pandas as pd
ROOT = Path("/content/drive/MyDrive/PharmacyDSS")
DATA = ROOT / "data/raw/supplier_performance.csv"
assert DATA.is_file(), f"Upload the dataset here: {DATA}"
OUT = ROOT / "runs/model3" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
OUT.mkdir(parents=True, exist_ok=False)
raw = pd.read_csv(DATA)
print("Shape:", raw.shape)
print("Missing values:", int(raw.isna().sum().sum()))
display(raw.head())


## 3. Write the reusable experiment script
This cell embeds the complete script. Keep it with the artifact bundle for review and reproduction.


In [ ]:
%%writefile /content/model3_supplier_scoring.py
"""Explainable supplier scorecard. Run from Colab with --data and --output.

Scores describe the supplied synthetic records, not actual supplier performance.
No supervised accuracy is claimed. Optional clustering requires scikit-learn.
"""
import argparse
import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd

WEIGHTS = {"on_time": 0.4, "fulfillment": 0.3, "price": 0.2, "quality": 0.1}


def score_suppliers(data, as_of, weights=None, window_days=365):
    weights = WEIGHTS if weights is None else weights
    if set(weights) != set(WEIGHTS) or any(v < 0 for v in weights.values()) or not np.isclose(sum(weights.values()), 1):
        raise ValueError("Weights must be nonnegative, sum to one and match the four criteria.")
    if window_days <= 0:
        raise ValueError("window_days must be positive")
    df = data.copy()
    required = ["order_id", "supplier_id", "supplier_name", "order_date", "promised_delivery_date", "actual_delivery_date", "order_quantity", "delivered_quantity", "rejected_units", "unit_price", "market_avg_price"]
    if not set(required).issubset(df):
        raise ValueError(f"Missing columns: {set(required) - set(df)}")
    if df[required].isna().any().any() or df.order_id.duplicated().any():
        raise ValueError("Missing required values or duplicate order IDs")
    for col in ["order_date", "promised_delivery_date", "actual_delivery_date"]:
        df[col] = pd.to_datetime(df[col], errors="raise")
    for col in ["order_quantity", "delivered_quantity", "rejected_units", "unit_price", "market_avg_price"]:
        df[col] = pd.to_numeric(df[col], errors="raise")
        if not np.isfinite(df[col]).all():
            raise ValueError(f"Nonfinite {col}")
    if not ((df.order_quantity > 0) & (df.delivered_quantity >= 0) & (df.rejected_units >= 0) & (df.rejected_units <= df.delivered_quantity) & (df.unit_price >= 0) & (df.market_avg_price > 0)).all():
        raise ValueError("Invalid quantities/prices")
    if ((df.actual_delivery_date < df.order_date) | (df.promised_delivery_date < df.order_date)).any():
        raise ValueError("Delivery dates precede order dates")
    end = pd.Timestamp(as_of)
    start = end - pd.Timedelta(days=window_days)
    # Completed-order retrospective score. Never use outcomes after the cutoff.
    df = df[(df.actual_delivery_date <= end) & (df.actual_delivery_date > start)].copy()
    if df.empty:
        raise ValueError("No completed orders in the scoring window")
    df["delay"] = (df.actual_delivery_date - df.promised_delivery_date).dt.days
    df["lead"] = (df.actual_delivery_date - df.order_date).dt.days
    df["accepted"] = df.delivered_quantity - df.rejected_units
    df["capped_delivered"] = np.minimum(df.delivered_quantity, df.order_quantity)
    df["price_ratio"] = df.unit_price / df.market_avg_price
    rows = []
    for sid, g in df.groupby("supplier_id"):
        # Fixed benchmark transform: zero premium -> 1; >=50% premium -> 0.
        # Discounts receive no bonus beyond 1. Weights/bounds are policy choices.
        price_score = np.clip(1 - max(0, g.price_ratio.median() - 1) / 0.5, 0, 1)
        delivered = g.delivered_quantity.sum()
        components = {
            "on_time": float((g.delay <= 0).mean()),
            "fulfillment": float(g.capped_delivered.sum() / g.order_quantity.sum()),
            "price": float(price_score),
            "quality": float(g.accepted.sum() / delivered) if delivered else 0.0,
        }
        score = 100 * sum(weights[k] * components[k] for k in weights)
        n = len(g)
        rows.append({"supplier_id": int(sid), "supplier_name": g.supplier_name.iloc[0],
                     "orders": n, "as_of": str(end.date()), "window_days": window_days,
                     **components, "score": score,
                     "tier": "insufficient_history" if n < 10 else "preferred" if score >= 85 else "acceptable" if score >= 70 else "review",
                     "avg_lead_time_days": float(g.lead.mean()),
                     "lead_time_std_days": float(g.lead.std(ddof=0)),
                     "median_price_ratio": float(g.price_ratio.median()),
                     **{f"contribution_{k}": 100 * weights[k] * components[k] for k in weights}})
    return pd.DataFrame(rows).sort_values(["score", "supplier_id"], ascending=[False, True]).reset_index(drop=True)


def run(data_path, output, as_of, clusters=False):
    path, out = Path(data_path), Path(output)
    out.mkdir(parents=True, exist_ok=True)
    raw = pd.read_csv(path)
    scores = score_suppliers(raw, as_of)
    scores.to_csv(out / "supplier_scores.csv", index=False)
    scenarios = []
    for criterion in WEIGHTS:
        for factor in [0.8, 1.2]:
            weights = {k: v * (factor if k == criterion else 1) for k, v in WEIGHTS.items()}
            total = sum(weights.values())
            weights = {k: v / total for k, v in weights.items()}
            frame = score_suppliers(raw, as_of, weights)
            frame["rank"] = frame.score.rank(ascending=False, method="min")
            frame["scenario"] = f"{criterion}_x{factor}"
            scenarios.append(frame[["supplier_id", "score", "rank", "scenario"]])
    sensitivity = pd.concat(scenarios, ignore_index=True)
    sensitivity.to_csv(out / "weight_sensitivity.csv", index=False)
    sensitivity.groupby("supplier_id")["rank"].agg(["min", "max"]).to_csv(out / "rank_ranges.csv")
    end = pd.Timestamp(as_of)
    previous = score_suppliers(raw, end - pd.Timedelta(days=180), window_days=180)
    recent = score_suppliers(raw, end, window_days=180)
    previous.merge(recent, on="supplier_id", suffixes=("_previous", "_recent")).to_csv(out / "period_comparison.csv", index=False)
    meta = {"model": "supplier_scorecard", "version": "1.0", "data_source": "synthetic",
            "as_of": as_of, "weights": WEIGHTS, "price_premium_zero_score": 0.5,
            "minimum_orders": 10, "tier_thresholds": {"preferred": 85, "acceptable": 70},
            "window_days": 365, "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
            "limitations": ["Completed orders only; open overdue orders require a separate operational alert.",
                            "Only 12 synthetic suppliers; clustering is exploratory.",
                            "Scores are not calibrated probabilities or real organisation ratings."]}
    if clusters:
        from sklearn.cluster import KMeans
        from sklearn.metrics import silhouette_score
        from sklearn.preprocessing import StandardScaler
        import joblib
        x = scores[list(WEIGHTS)]
        scaler = StandardScaler().fit(x)
        z = scaler.transform(x)
        candidates = []
        for k in range(2, min(5, len(scores))):
            estimator = KMeans(n_clusters=k, random_state=42, n_init=20).fit(z)
            if 1 < len(set(estimator.labels_)) < len(scores):
                candidates.append((silhouette_score(z, estimator.labels_), k, estimator))
        if not candidates:
            raise ValueError("No meaningful cluster solution")
        silhouette, k, estimator = max(candidates, key=lambda c: c[0])
        scores["cluster_id"] = estimator.labels_
        scores.to_csv(out / "exploratory_clusters.csv", index=False)
        joblib.dump({"scaler": scaler, "model": estimator, "features": list(WEIGHTS)}, out / "exploratory_clusters.joblib")
        meta["clustering"] = {"k": k, "silhouette": silhouette, "note": "Cluster IDs do not define supplier quality tiers."}
    (out / "scorecard_policy.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")
    print(scores[["supplier_id", "score", "tier"]].to_string(index=False))
    return scores


if __name__ == "__main__":
    p = argparse.ArgumentParser()
    p.add_argument("--data", required=True)
    p.add_argument("--output", required=True)
    p.add_argument("--as-of", default="2024-12-31")
    p.add_argument("--clusters", action="store_true")
    a = p.parse_args()
    run(a.data, a.output, a.as_of, a.clusters)


## 4. Run the scorecard
The cutoff excludes deliveries not yet observed. Weights and tier thresholds are proposed policy choices, not learned truth. The score only covers completed orders; operational overdue-order alerts remain separate. Set CLUSTERS to True only for exploratory analysis of these 12 synthetic suppliers.


In [ ]:
AS_OF = "2024-12-31"
CLUSTERS = False
command = [sys.executable, "/content/model3_supplier_scoring.py", "--data", str(DATA), "--output", str(OUT), "--as-of", AS_OF]
if CLUSTERS:
    command.append("--clusters")
subprocess.run(command, check=True)


## 5. Review score contributions and sensitivity
Large rank ranges indicate dependence on the proposed weights. Public charts use supplier IDs so simulated results are not mistaken for actual company ratings.


In [ ]:
import matplotlib.pyplot as plt
scores = pd.read_csv(OUT / "supplier_scores.csv")
display(scores)
display(pd.read_csv(OUT / "rank_ranges.csv"))
columns = ["contribution_on_time", "contribution_fulfillment", "contribution_price", "contribution_quality"]
scores.set_index("supplier_id")[columns].plot.barh(stacked=True, figsize=(10, 6))
plt.title("Synthetic supplier score contributions")
plt.xlabel("Score points (0–100)")
plt.tight_layout()
plt.savefig(OUT / "score_contributions.png", dpi=150)
plt.show()


## 6. Save the environment and script
The exported outputs are candidates. Add reviewed feature schemas, golden examples and a model card before Django promotion. Only load your own trusted joblib artifacts in a compatible environment.


In [ ]:
import shutil
lock = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
(OUT / "requirements-lock.txt").write_text(lock)
shutil.copyfile("/content/model3_supplier_scoring.py", OUT / "model3_supplier_scoring.py")
print("Results saved:", OUT)
print("\n".join(p.name for p in sorted(OUT.iterdir())))
